# WLASL 10-Word Landmark Model

Train one landmark sequence model for 10 WLASL words only:

`book`, `finish`, `go`, `good`, `help`, `like`, `mother`, `what`, `who`, `yes`

Total classes: 10. Alphabet and digit classes are intentionally excluded.


In [ ]:
# CELL 1 - Install and download MediaPipe task models
!pip install mediapipe yt-dlp --quiet

import os
import urllib.request
import mediapipe as mp

print('MediaPipe', mp.__version__)

HAND_MODEL = '/kaggle/working/hand_landmarker.task'
POSE_MODEL = '/kaggle/working/pose_landmarker.task'

for path, url in [
    (HAND_MODEL, 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'),
    (POSE_MODEL, 'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task'),
]:
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
        print(f'Downloaded {os.path.basename(path)}')

print('Ready')


In [ ]:
# CELL 2 - Imports
import json, os, cv2, glob, random, pickle, subprocess, tempfile
import numpy as np
import pandas as pd
import tensorflow as tf
import mediapipe as mp
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

print('Imports OK')
print('TF:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# CELL 3 - Config
SEQ_LEN = 20
FEATURE_DIM = 225
BATCH_SIZE = 16
EPOCHS = 100
MAX_PER_CLASS = 150

SAVE_DIR = '/kaggle/working/'
LANDMARKS_DIR = '/kaggle/working/landmarks_words_only/'
os.makedirs(LANDMARKS_DIR, exist_ok=True)


# Existing word datasets
JSON_PATH = '/kaggle/input/datasets/sttaseen/wlasl2000-resized/wlasl-complete/WLASL_v0.3.json'
VIDEOS_1 = '/kaggle/input/datasets/sttaseen/wlasl2000-resized/wlasl-complete/videos'
VIDEOS_2 = '/kaggle/input/datasets/risangbaskoro/wlasl-processed/videos'
TWINTALK_DIR = '/kaggle/input/datasets/jomanahalshangiti/twintalk-asl-data/new_data_900/data900'
MUTEMOTION_NPZ = '/kaggle/input/datasets/abd0kamel/mutemotion-output/landmarks_V3.npz'
MUTEMOTION_JSON = '/kaggle/input/datasets/abd0kamel/mutemotion-output/WLASL_parsed_data.json'
ASLCITIZEN_VIDS = '/kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/videos'
ASLCITIZEN_SPLIT = '/kaggle/input/datasets/abd0kamel/asl-citizen/ASL_Citizen/splits'
WASEEM_DIR = '/kaggle/input/datasets/waseemnagahhenes/sign-language-dataset-wlasl-videos/dataset/SL'
ASTHASL_DIR = '/kaggle/input/datasets/asthalochanmohanta/american-sign-language-asl/dataset_v2'
MSASL_DIR = '/kaggle/input/datasets/saurabhshahane/american-sign-language-dataset'

SELECTED_10 = ['book','like','mother','finish','help','go','yes','who','what','good']
WORD_CLASSES = sorted(SELECTED_10)

selected_words = WORD_CLASSES
NUM_CLASSES = len(selected_words)
word_to_index = {w: i for i, w in enumerate(selected_words)}
index_to_word = {i: w for w, i in word_to_index.items()}


print(f'Classes ({NUM_CLASSES}):')
print(selected_words)
print(f'MAX_PER_CLASS: {MAX_PER_CLASS}')


In [ ]:
# CELL 4 - MediaPipe extractor for video frames
_hand_det = mp_vision.HandLandmarker.create_from_options(
    mp_vision.HandLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=HAND_MODEL),
        num_hands=2,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
        running_mode=mp_vision.RunningMode.IMAGE,
    )
)
_pose_det = mp_vision.PoseLandmarker.create_from_options(
    mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=POSE_MODEL),
        min_pose_detection_confidence=0.3,
        min_tracking_confidence=0.3,
        running_mode=mp_vision.RunningMode.IMAGE,
    )
)

def frame_to_landmarks(frame_bgr):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    hr = _hand_det.detect(mp_img)
    lh = np.zeros(63, dtype=np.float32)
    rh = np.zeros(63, dtype=np.float32)
    for i, handedness in enumerate(hr.handedness):
        if i >= len(hr.hand_landmarks):
            continue
        lms = np.array([[lm.x, lm.y, lm.z] for lm in hr.hand_landmarks[i]], dtype=np.float32).flatten()
        if handedness[0].category_name == 'Left':
            lh = lms
        else:
            rh = lms

    pr = _pose_det.detect(mp_img)
    pose = np.array([[lm.x, lm.y, lm.z] for lm in pr.pose_landmarks[0]], dtype=np.float32).flatten() \
           if pr.pose_landmarks else np.zeros(99, dtype=np.float32)

    return np.concatenate([lh, rh, pose]).astype(np.float32)

def extract_landmarks(video_path, seq_len=SEQ_LEN):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 2:
        cap.release()
        return None
    indices = np.linspace(0, total - 1, seq_len, dtype=int)
    sequence = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            sequence.append(np.zeros(FEATURE_DIM, dtype=np.float32))
            continue
        try:
            sequence.append(frame_to_landmarks(frame))
        except Exception:
            sequence.append(np.zeros(FEATURE_DIM, dtype=np.float32))
    cap.release()
    return np.array(sequence, dtype=np.float32) if len(sequence) == seq_len else None

print('Extractor ready')


In [ ]:
# CELL 5 - Save/count helpers
def count_per_class():
    counts = Counter()
    for f in glob.glob(f'{LANDMARKS_DIR}*.npy'):
        try:
            label = int(Path(f).stem.rsplit('_', 1)[-1])
            counts[label] += 1
        except Exception:
            pass
    return counts

def save_sequence(seq, output_path):
    if seq is None or seq.shape != (SEQ_LEN, FEATURE_DIM):
        return False
    np.save(output_path, seq.astype(np.float32))
    return True

def save_video_if_room(video_path, label, prefix=''):
    counts = count_per_class()
    if counts[label] >= MAX_PER_CLASS:
        return False
    vid_id = Path(video_path).stem
    npy_path = os.path.join(LANDMARKS_DIR, f'{prefix}{vid_id}_{label}.npy')
    if os.path.exists(npy_path):
        return True
    return save_sequence(extract_landmarks(video_path), npy_path)

print('Helpers ready')


In [ ]:
# CELL 6 - Source A/B/C/D: WLASL, Waseem, AST-ASL, ASL Citizen
print('=== Source A: WLASL videos ===')
if os.path.exists(JSON_PATH):
    with open(JSON_PATH) as f:
        data_full = json.load(f)
    saved = skipped = 0
    for entry in data_full:
        gloss = entry['gloss'].lower().strip()
        if gloss not in word_to_index:
            continue
        label = word_to_index[gloss]
        for inst in entry['instances']:
            vid_id = inst['video_id']
            vid_path = None
            for vdir in [VIDEOS_1, VIDEOS_2, TWINTALK_DIR]:
                p = os.path.join(vdir, f'{vid_id}.mp4')
                if os.path.exists(p):
                    vid_path = p
                    break
            if vid_path is None:
                skipped += 1
                continue
            save_video_if_room(vid_path, label, 'wlasl_')
            saved += 1
    print(f'WLASL: {saved} processed | {skipped} missing')
else:
    print('Skip WLASL: JSON not found')

print('=== Source B: waseem-wlasl ===')
saved = 0
for word in SELECTED_10:
    word_dir = os.path.join(WASEEM_DIR, word)
    if not os.path.exists(word_dir):
        continue
    label = word_to_index[word]
    for vid_file in os.listdir(word_dir):
        if vid_file.endswith('.mp4'):
            save_video_if_room(os.path.join(word_dir, vid_file), label, 'waseem_')
            saved += 1
print(f'waseem-wlasl: {saved} processed')

print('=== Source C: ast-asl ===')
saved = 0
for split in ['train', 'val', 'test']:
    split_path = os.path.join(ASTHASL_DIR, split)
    if not os.path.exists(split_path):
        continue
    for word in SELECTED_10:
        word_dir = os.path.join(split_path, word)
        if not os.path.exists(word_dir):
            continue
        label = word_to_index[word]
        for vid_file in os.listdir(word_dir):
            if vid_file.endswith('.mp4'):
                save_video_if_room(os.path.join(word_dir, vid_file), label, f'ast_{split}_')
                saved += 1
print(f'ast-asl: {saved} processed')

print('=== Source D: ASL Citizen ===')
if os.path.exists(ASLCITIZEN_SPLIT):
    dfs = [pd.read_csv(os.path.join(ASLCITIZEN_SPLIT, f)) for f in os.listdir(ASLCITIZEN_SPLIT) if f.endswith('.csv')]
    if dfs:
        df = pd.concat(dfs)
        df['gloss_lower'] = df['Gloss'].str.lower().str.strip()
        saved = skipped = 0
        for word in SELECTED_10:
            label = word_to_index[word]
            matches = df[df['gloss_lower'] == word]
            for _, row in matches.iterrows():
                vid_path = os.path.join(ASLCITIZEN_VIDS, row['Video file'])
                if not os.path.exists(vid_path):
                    skipped += 1
                    continue
                save_video_if_room(vid_path, label, 'aslc_')
                saved += 1
        print(f'ASL Citizen: {saved} processed | {skipped} missing')
print(f'Total .npy: {len(glob.glob(LANDMARKS_DIR + "*.npy"))}')


In [ ]:
# CELL 7 - Source E/F: MS-ASL and MuteMotion
print('=== Source E: MS-ASL ===')
classes_path = os.path.join(MSASL_DIR, 'MSASL_classes.json')
if os.path.exists(classes_path):
    with open(classes_path) as f:
        classes = json.load(f)
    MSASL_MAP = {classes[i].lower(): i for i in range(len(classes)) if classes[i].lower() in SELECTED_10}
    all_msasl = []
    for split in ['MSASL_train.json', 'MSASL_val.json', 'MSASL_test.json']:
        p = os.path.join(MSASL_DIR, split)
        if os.path.exists(p):
            with open(p) as f:
                all_msasl += json.load(f)
    saved = skipped = errors = 0
    per_class = count_per_class()
    for idx, sample in enumerate(all_msasl):
        word = next((w for w, lid in MSASL_MAP.items() if lid == sample['label']), None)
        if word is None:
            continue
        label = word_to_index[word]
        if per_class[label] >= MAX_PER_CLASS:
            continue
        npy_path = os.path.join(LANDMARKS_DIR, f'msasl_{idx}_{label}.npy')
        if os.path.exists(npy_path):
            saved += 1
            per_class[label] += 1
            continue
        try:
            with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
                tmp_path = tmp.name
            start = float(sample.get('start_time', 0))
            end = float(sample.get('end_time', start + 3))
            subprocess.run([
                'yt-dlp', '-q', '--no-warnings',
                '-f', 'mp4/best[height<=360]/best',
                '--download-sections', f'*{start:.1f}-{end:.1f}',
                '--force-keyframes-at-cuts',
                '-o', tmp_path, sample['url']
            ], timeout=25, capture_output=True)
            if os.path.exists(tmp_path) and os.path.getsize(tmp_path) > 1000:
                ok = save_sequence(extract_landmarks(tmp_path), npy_path)
                saved += int(ok)
                errors += int(not ok)
                per_class[label] += int(ok)
            else:
                skipped += 1
        except Exception:
            skipped += 1
        finally:
            if 'tmp_path' in locals() and os.path.exists(tmp_path):
                os.remove(tmp_path)
    print(f'MS-ASL: {saved} saved | {skipped} unavailable | {errors} errors')
else:
    print('Skip MS-ASL: classes file not found')

print('=== Source F: MuteMotion NPZ ===')
if os.path.exists(MUTEMOTION_NPZ) and os.path.exists(MUTEMOTION_JSON):
    mm_npz = np.load(MUTEMOTION_NPZ, allow_pickle=True)
    with open(MUTEMOTION_JSON) as f:
        mm_data = json.load(f)
    def npz_to_sequence(arr, seq_len=SEQ_LEN):
        N = arr.shape[0]
        if N < 2:
            return None
        indices = np.linspace(0, N - 1, seq_len, dtype=int)
        seq = arr[indices]
        lh = seq[:, 21:42, :].reshape(seq_len, 63).astype(np.float32)
        rh = seq[:, 0:21, :].reshape(seq_len, 63).astype(np.float32)
        pose = seq[:, 42:75, :].reshape(seq_len, 99).astype(np.float32)
        return np.concatenate([lh, rh, pose], axis=1)
    added = skipped = 0
    per_class = count_per_class()
    for key in list(mm_npz.keys()):
        idx = int(key)
        if idx >= len(mm_data):
            continue
        gloss = mm_data[idx]['gloss'].lower().strip()
        if gloss not in word_to_index:
            skipped += 1
            continue
        label = word_to_index[gloss]
        if per_class[label] >= MAX_PER_CLASS:
            continue
        npy_path = os.path.join(LANDMARKS_DIR, f'mm_{key}_{label}.npy')
        if os.path.exists(npy_path):
            added += 1
            per_class[label] += 1
            continue
        if save_sequence(npz_to_sequence(mm_npz[key]), npy_path):
            added += 1
            per_class[label] += 1
    print(f'MuteMotion: {added} added | {skipped} skipped')
else:
    print('Skip MuteMotion: files not found')
print(f'Total .npy: {len(glob.glob(LANDMARKS_DIR + "*.npy"))}')


In [ ]:
# CELL 9 - Check per-class counts
all_files = sorted(glob.glob(f'{LANDMARKS_DIR}*.npy'))
all_labels = []
for f in all_files:
    try:
        all_labels.append(int(Path(f).stem.rsplit('_', 1)[-1]))
    except Exception:
        all_labels.append(-1)

all_files = [f for f, l in zip(all_files, all_labels) if l in range(NUM_CLASSES)]
all_labels = [l for l in all_labels if l in range(NUM_CLASSES)]

counts = Counter(all_labels)
print(f'Total: {len(all_files)} samples | {len(counts)} / {NUM_CLASSES} classes')
print(f'\n{"Class":20} {"Count":>6} {"Status":>8}')
print('-' * 38)
for label in range(NUM_CLASSES):
    c = counts[label]
    status = 'OK' if c >= 80 else 'LOW' if c >= 30 else 'BAD'
    print(f'{index_to_word[label]:20} {c:>6} {status:>8}')

missing = [index_to_word[i] for i in range(NUM_CLASSES) if counts[i] == 0]
if missing:
    raise RuntimeError(f'Missing classes: {missing}')


In [ ]:
# CELL 10 - Normalize + balance + split
def normalize_sequence(seq):
    seq = seq.copy()
    for t in range(seq.shape[0]):
        lh = seq[t, 0:63].reshape(21, 3)
        if np.any(lh != 0):
            lh = lh - lh[0]
            s = np.max(np.linalg.norm(lh, axis=1)) + 1e-8
            seq[t, 0:63] = (lh / s).flatten()
        rh = seq[t, 63:126].reshape(21, 3)
        if np.any(rh != 0):
            rh = rh - rh[0]
            s = np.max(np.linalg.norm(rh, axis=1)) + 1e-8
            seq[t, 63:126] = (rh / s).flatten()
        pose = seq[t, 126:225].reshape(33, 3)
        if np.any(pose != 0):
            pose = pose - pose[0]
            sw = np.linalg.norm(pose[11] - pose[12]) + 1e-8
            seq[t, 126:225] = (pose / sw).flatten()
    return seq.astype(np.float32)

min_count = min(counts.values())
print(f'Balancing at {min_count} samples per class')

paired = list(zip(all_files, all_labels))
random.shuffle(paired)
class_seen = Counter()
bal_files, bal_labels = [], []
for f, l in paired:
    if class_seen[l] < min_count:
        bal_files.append(f)
        bal_labels.append(l)
        class_seen[l] += 1

X, y = [], []
for f, l in zip(bal_files, bal_labels):
    try:
        seq = np.load(f)
        if seq.shape == (SEQ_LEN, FEATURE_DIM):
            X.append(normalize_sequence(seq))
            y.append(l)
    except Exception:
        pass

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print(f'X: {X.shape} | y: {y.shape}')

train_X, val_X, train_y, val_y = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(train_X)} | Val: {len(val_X)}')


In [ ]:
# CELL 11 - Augmentation + tf.data pipeline
def augment_sequence(seq):
    seq = seq.copy()
    if np.random.rand() < 0.5:
        speed = np.random.uniform(0.75, 1.25)
        new_len = max(2, int(SEQ_LEN * speed))
        indices = np.linspace(0, SEQ_LEN - 1, new_len)
        seq_i = np.array([seq[int(i)] * (1 - i % 1) + seq[min(int(i) + 1, SEQ_LEN - 1)] * (i % 1) for i in indices])
        seq = seq_i[np.linspace(0, len(seq_i) - 1, SEQ_LEN).astype(int)]
    if np.random.rand() < 0.6:
        seq = seq + np.random.normal(0, 0.015, seq.shape).astype(np.float32)
    if np.random.rand() < 0.4:
        lh = seq[:, :63].copy()
        rh = seq[:, 63:126].copy()
        seq[:, :63] = rh
        seq[:, 63:126] = lh
        seq[:, 0::3] = -seq[:, 0::3]
    if np.random.rand() < 0.4:
        seq[:, :126] *= np.random.uniform(0.8, 1.2)
    if np.random.rand() < 0.4:
        shift = np.random.uniform(-0.1, 0.1, 3).astype(np.float32)
        seq[:, 0:63:3] += shift[0]
        seq[:, 1:63:3] += shift[1]
        seq[:, 63:126:3] += shift[0]
        seq[:, 64:126:3] += shift[1]
    if np.random.rand() < 0.3:
        for d in np.random.choice(SEQ_LEN, np.random.randint(1, 4), replace=False):
            seq[d] = np.zeros(FEATURE_DIM, dtype=np.float32)
    return seq.astype(np.float32)

def augment_tf(x, y):
    x = tf.numpy_function(augment_sequence, [x], tf.float32)
    x.set_shape((SEQ_LEN, FEATURE_DIM))
    return x, y

train_y_oh = tf.keras.utils.to_categorical(train_y, NUM_CLASSES)
val_y_oh = tf.keras.utils.to_categorical(val_y, NUM_CLASSES)

train_ds = tf.data.Dataset.from_tensor_slices((train_X, train_y_oh))
train_ds = train_ds.shuffle(len(train_X)).map(augment_tf).batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)
val_ds = tf.data.Dataset.from_tensor_slices((val_X, val_y_oh))
val_ds = val_ds.batch(BATCH_SIZE).repeat().prefetch(tf.data.AUTOTUNE)

STEPS_PER_EPOCH = max(1, len(train_X) // BATCH_SIZE)
VALIDATION_STEPS = max(1, len(val_X) // BATCH_SIZE)
print(f'Pipeline ready | steps:{STEPS_PER_EPOCH} | val_steps:{VALIDATION_STEPS}')


In [ ]:
# CELL 12 - BiLSTM model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQ_LEN, FEATURE_DIM)),
    tf.keras.layers.LayerNormalization(),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=False)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax'),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()
print(f'Model ready - {NUM_CLASSES} classes | Random baseline: {100 / NUM_CLASSES:.1f}%')


In [ ]:
# CELL 13 - Train
history = model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    validation_steps=VALIDATION_STEPS,
    epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(f'{SAVE_DIR}model_landmarks.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=1),
    ],
    verbose=1,
)
print('Training done')


In [ ]:
# CELL 14 - Save mapping + H5 + weights
best = tf.keras.models.load_model(f'{SAVE_DIR}model_landmarks.keras')

mapping = {
    'word_to_index': word_to_index,
    'index_to_word': {str(k): v for k, v in index_to_word.items()},
    'selected_words': list(index_to_word.values()),
    'num_classes': NUM_CLASSES,
    'seq_len': SEQ_LEN,
    'feature_dim': FEATURE_DIM,
    'normalized': True,
    'architecture': 'wlasl_words_only',
}
with open(f'{SAVE_DIR}mapping.json', 'w') as f:
    json.dump(mapping, f, indent=2)

with open(f'{SAVE_DIR}model_weights.pkl', 'wb') as f:
    pickle.dump(best.get_weights(), f)

best.save(f'{SAVE_DIR}model_landmarks.h5')

print('Download from Kaggle output:')
for f in ['model_landmarks.keras', 'model_landmarks.h5', 'model_weights.pkl', 'mapping.json']:
    path = f'{SAVE_DIR}{f}'
    exists = os.path.exists(path)
    size = os.path.getsize(path) / 1024 / 1024 if exists else 0
    print(f"{'OK' if exists else 'NO'} {f:24} {size:.1f} MB")


In [ ]:
# CELL 15 - Accuracy check
preds_all = best.predict(val_X, verbose=0)
p = np.argmax(preds_all, axis=1)
t = val_y

correct = np.sum(p == t)
total = len(t)
top3_ok = sum(t[i] in np.argsort(preds_all[i])[-3:] for i in range(total))

print(f'Val Accuracy  : {correct / total * 100:.2f}% ({correct}/{total})')
print(f'Top-3 Accuracy: {top3_ok / total * 100:.2f}%')
print(f'Random baseline: {100 / NUM_CLASSES:.1f}%')
print('Sample predictions:')
for i in range(min(20, total)):
    pred = index_to_word[p[i]]
    real = index_to_word[t[i]]
    top3w = [index_to_word[j] for j in np.argsort(preds_all[i])[-3:][::-1]]
    print(f"  {'OK' if pred == real else 'NO'} Real:{real:12} Pred:{pred:12} Top3:{top3w}")


In [ ]:
# CELL 16 - Training curves and confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(history.history['accuracy'], label='Train', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val', linewidth=2)
ax1.axhline(y=1 / NUM_CLASSES, color='r', linestyle='--', label=f'Random ({100 / NUM_CLASSES:.1f}%)')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)
ax2.plot(history.history['loss'], label='Train', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val', linewidth=2)
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(alpha=0.3)
plt.suptitle(f'WLASL Words Only ({NUM_CLASSES} classes)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}training_curve.png', dpi=150)
plt.show()

labels = list(range(NUM_CLASSES))
words = [index_to_word[i] for i in labels]
cm = confusion_matrix(t, p, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm_norm, cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set_xticks(labels)
ax.set_yticks(labels)
ax.set_xticklabels(words, rotation=90, ha='center', fontsize=8)
ax.set_yticklabels(words, fontsize=8)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix - {NUM_CLASSES} classes')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}confusion_matrix.png', dpi=150)
plt.show()
